# 6. Loop dmr motif

Part of the **[Fig. 5 chapter](fig5.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

_No explicit file reads detected (data may be read via variables or shell cells)._


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [1]:
from glob import glob
import os
import anndata
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
from scipy.stats import zscore

from ALLCools.clustering import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.plot import *
from sklearn.metrics import pairwise_distances, roc_auc_score
from sklearn.preprocessing import normalize

mpl.style.use("default")

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = "Helvetica"

import warnings
warnings.filterwarnings("ignore")


In [2]:
indir = f'{ENTEX_ROOT}/'
dmrdir = f'{indir}analysis/DMR/'
loopdir = f'{indir}loop/majortype/'
outdir = f'{indir}analysis/loop_dmr_motif/'


In [4]:
# ctx_db_fname = f'{dmrdir}merged_dmr_db/merged_dmr_slop1kb.regions_vs_motifs.rankings.feather'
# dem_db_fname = f'{dmrdir}merged_dmr_db/merged_dmr_slop1kb.regions_vs_motifs.scores.feather'


In [5]:
# score = pd.read_feather(dem_db_fname)
# score

,chr10:100000953-100000961,chr10:100001679-100002245,chr10:100003300-100003767,chr10:100004641-100005072,chr10:100005550-100005711,chr10:100005979-100006587,chr10:100006892-100007008,chr10:100007313-100007734,chr10:100007991-100008955,chr10:100010619-100010667,...,chr9:99977538-99978235,chr9:9997974-9998312,chr9:99982260-99982626,chr9:99985900-99985916,chr9:9998593-9999196,chr9:99994951-99994988,chr9:9999586-10000062,chr9:99997862-99997949,chr9:99998235-99999140,motifs
0,0.0,0.160,0.000,2.190,4.0000,0.000,0.000,0.000,0.0000,0.000,...,0.00000,0.0000,2.7400,0.000,3.2500,0.000,2.330,0.000,4.3700,bergman__Adf1
1,0.0,0.147,9.270,2.850,0.0000,3.300,0.000,0.337,0.2460,0.000,...,0.20100,0.1220,0.1180,0.000,0.0368,0.145,0.114,0.000,0.0186,bergman__Aef1
2,0.0,0.191,4.520,0.287,0.1590,0.113,0.131,0.117,0.0756,0.000,...,4.65000,0.3320,0.0000,0.000,0.1180,0.000,0.201,0.000,4.6700,bergman__Hr46
3,0.0,0.000,0.000,0.000,0.0000,0.000,0.000,0.000,0.0000,0.000,...,0.00000,0.0000,0.0000,0.000,0.0000,0.000,0.000,0.000,0.0000,bergman__Kr
4,0.0,0.325,3.320,3.480,0.0000,3.290,0.000,3.680,0.1950,0.000,...,0.11300,0.0762,0.0000,0.000,3.6400,0.000,3.790,0.166,2.9100,bergman__Su_H_
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10244,0.0,3.030,2.650,0.000,2.7600,0.000,2.030,3.360,3.8800,2.260,...,0.00745,3.0600,0.0203,0.000,2.7900,0.000,0.000,0.000,11.8000,yetfasco__YPR008W_1425
10245,0.0,3.480,2.390,1.080,1.7300,4.970,1.090,1.970,2.4600,0.589,...,3.55000,5.5300,1.6000,0.000,1.5700,0.000,0.783,0.000,3.6500,yetfasco__YPR009W_2236
10246,0.0,0.000,0.000,2.100,0.0000,0.000,0.000,0.000,0.0000,0.000,...,1.39000,0.0000,1.5900,0.000,0.0000,0.000,0.000,0.000,0.0000,yetfasco__YPR054W_1875
10247,0.0,2.280,0.499,0.490,0.0427,2.420,0.211,2.150,3.5000,0.289,...,1.17000,0.4000,2.4100,0.488,0.5250,0.000,1.550,1.470,0.5000,yetfasco__YPR065W_1396


In [6]:
# rank = pd.read_feather(ctx_db_fname)
# rank


,chr10:100000953-100000961,chr10:100001679-100002245,chr10:100003300-100003767,chr10:100004641-100005072,chr10:100005550-100005711,chr10:100005979-100006587,chr10:100006892-100007008,chr10:100007313-100007734,chr10:100007991-100008955,chr10:100010619-100010667,...,chr9:99977538-99978235,chr9:9997974-9998312,chr9:99982260-99982626,chr9:99985900-99985916,chr9:9998593-9999196,chr9:99994951-99994988,chr9:9999586-10000062,chr9:99997862-99997949,chr9:99998235-99999140,motifs
0,486783,260023,351572,170215,38161,1380185,913022,547266,630594,920779,...,849617,1121566,113829,627519,70786,1340733,155655,773621,30882,bergman__Adf1
1,1430362,485306,3650,284624,1204526,259915,1028390,323163,357766,1080975,...,396008,553475,566479,1430808,886770,489009,580628,1303225,948536,bergman__Aef1
2,1334825,553790,74736,393761,629372,775212,713955,758953,928132,1353101,...,56751,346583,1212642,1391036,755315,1394674,532279,1314125,54773,bergman__Hr46
3,470014,337413,1301488,435391,946829,282733,700044,696182,1290127,1167551,...,650399,1258293,844281,137706,803858,670526,223425,846298,436567,bergman__Kr
4,1136763,646010,407007,341707,1378618,417820,1291473,269931,715252,1223087,...,814111,882039,1178608,1272470,282253,1182956,239618,743839,537104,bergman__Su_H_
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10244,1272249,314376,424919,1231945,392799,1027851,577368,237816,161777,530243,...,803523,305909,783283,1077709,381958,1318906,1164090,826648,5328,yetfasco__YPR008W_1425
10245,1320267,222793,443445,869302,646574,88212,865735,566130,424771,1048797,...,213519,62524,689755,1319715,698657,1269703,978143,1402808,200421,yetfasco__YPR009W_2236
10246,1064736,830949,1168886,116706,803468,585823,521375,534861,1333105,1242986,...,178018,812907,158730,481060,475737,739574,815349,917062,914831,yetfasco__YPR054W_1875
10247,1403716,330497,944062,948549,1190765,297144,1099773,363173,107881,1057308,...,649093,996121,299837,949438,930549,1231691,523607,550359,943255,yetfasco__YPR065W_1396


In [ ]:
# import time
# from scipy.stats import rankdata

# start_time = time.time()
# for i in range(37):
#     if i==7:
#         continue
#     ct = f'c{i}'
#     print(ct)
#     os.makedirs(f'{outdir}{ct}/', exist_ok=True)
#     cmd = f'bedtools intersect -wa -a {dmrdir}merged_dmr.bed -b {dmrdir}overlap_atac/majortype_36groups/{ct}_merged.bed > {outdir}{ct}/dmr.bed'
#     os.system(cmd)
#     tmp = pd.read_csv(f'{outdir}{ct}/dmr.bed', sep='\t', names=['chrom','start','end'])
#     seldmr = tmp['chrom'].astype(str) + ':' + tmp['start'].astype(str) + '-' + tmp['end'].astype(str)
#     seldmr = score.columns.isin(seldmr)
#     # seldmr[-1] = True
#     score_tmp = score.loc[:, seldmr]
#     rank_tmp = rank.loc[:, seldmr]
#     print('Filter DMR', time.time()-start_time)
#     if i>0:
#         score_tmp['motifs'] = score['motifs']
#         score_tmp.to_feather(f'{outdir}{ct}/regions_vs_motifs.scores.feather')
#         print('Write score', time.time()-start_time)
#     rank_tmp = rankdata(rank_tmp, axis=1)
#     rank_tmp = pd.DataFrame(rank_tmp, index=rank.index, columns=score_tmp.columns)
#     print('Compute rank', time.time()-start_time)
#     rank_tmp['motifs'] = rank['motifs']
#     rank_tmp.to_feather(f'{outdir}{ct}/regions_vs_motifs.rankings.feather')
#     print('Write rank', time.time()-start_time)
    

c0
ERROR! Session/line number was not unique in database. History logging moved to new session 3476
Filter DMR 21.95212984085083


In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ## DMR given size slop1k createdb
# ct=c0
# REGION_BED=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR/overlap_atac/majortype_36groups/${ct}_merged.bed
# GENOME_FASTA=/large_storage/zhoulab/ref/hg38/fasta/hg38.fa
# CHROMSIZES=/large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes
# DATABASE_PREFIX=merge_dmr
# SCRIPT_DIR=/home/zhoujt/software/create_cisTarget_databases
# OUT_DIR=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif/${ct}
# mkdir $OUT_DIR
# ${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
#         ${GENOME_FASTA} \
#         ${CHROMSIZES} \
#         ${REGION_BED} \
#         ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa \
#         1000 \
#         yes

# export PATH=$PATH:~/software/
# REF_DIR=/large_storage/zhoulab/ref/aertslab_motif_colleciton
# CBDIR=${REF_DIR}/v10nr_clust_public/singletons
# FASTA_FILE=${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa
# MOTIF_LIST=${REF_DIR}/motifs.txt

# ${SCRIPT_DIR}/create_cistarget_motif_databases.py \
#     -f ${FASTA_FILE} \
#     -M ${CBDIR} \
#     -m ${MOTIF_LIST} \
#     -o ${OUT_DIR}/${DATABASE_PREFIX} \
#     --bgpadding 1000 \
#     -t 32



In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ## DMR 1kb createdb
# ct=c0
# REGION_BED=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR/overlap_atac/majortype_36groups/${ct}_merged.bed
# GENOME_FASTA=/large_storage/zhoulab/ref/hg38/fasta/hg38.fa
# CHROMSIZES=/large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes
# DATABASE_PREFIX=merge_dmr
# SCRIPT_DIR=/home/zhoujt/software/create_cisTarget_databases
# OUT_DIR=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif/${ct}
# mkdir $OUT_DIR
# awk '{printf("%s\t%d\t%d\n",$1,($2+$3)/2,($2+$3)/2)}' $REGION_BED | bedtools slop -i - -b 500 -g /large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes > ${OUT_DIR}/${DATABASE_PREFIX}.1kb.bed
# ${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
#         ${GENOME_FASTA} \
#         ${CHROMSIZES} \
#         ${OUT_DIR}/${DATABASE_PREFIX}.1kb.bed \
#         ${OUT_DIR}/${DATABASE_PREFIX}.1kb.fa \
#         0 \
#         yes

# export PATH=$PATH:~/software/
# REF_DIR=/large_storage/zhoulab/ref/aertslab_motif_colleciton
# CBDIR=${REF_DIR}/v10nr_clust_public/singletons
# FASTA_FILE=${OUT_DIR}/${DATABASE_PREFIX}.1kb.fa
# MOTIF_LIST=${REF_DIR}/motifs.txt

# ${SCRIPT_DIR}/create_cistarget_motif_databases.py \
#     -f ${FASTA_FILE} \
#     -M ${CBDIR} \
#     -m ${MOTIF_LIST} \
#     -o ${OUT_DIR}/${DATABASE_PREFIX}.1kb \
#     --bgpadding 0 \
#     -t 32



In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ## summit/loop DMR enriched motif

# for i in 1 3; do (
# ct=c${i}
# loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
# dmrdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR
# outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif
# mkdir -p ${outdir}/${ct}/bed/loop_dmr/ ${outdir}/${ct}/bed/summit_dmr/
# # awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# # awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed

# scenicplus grn_inference motif_enrichment_cistarget \
#     --region_set_folder ${outdir}/${ct}/bed \
#     --cistarget_db_fname ${outdir}/${ct}/merge_dmr.1kb.regions_vs_motifs.rankings.feather \
#     --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.1kb.hdf5 \
#     --temp_dir ${outdir}/${ct}/tmp \
#     --species "homo_sapiens" \
#     --fr_overlap_w_ctx_db 0.4 \
#     --auc_threshold 0.005 \
#     --nes_threshold 3.0 \
#     --rank_threshold 0.05 \
#     --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
#     --annotation_version "v10nr_clust" \
#     --motif_similarity_fdr 0.001 \
#     --orthologous_identity_threshold 0.0 \
#     --annotations_to_use "Direct_annot Orthology_annot" \
#     --write_html \
#     --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.1kb.html \
#     --n_cpu 32

# # for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.1kb.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; echo; done | cut -f6 | sort -k1,1 -u | less
# ) done



In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ## summit/loop DMR peak enriched motif

# for i in `seq 35 36`; do (
# ct=c${i}
# loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
# dmrdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR
# outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif
# mkdir -p ${outdir}/${ct}/bed/loop_dmr/ ${outdir}/${ct}/bed/summit_dmr/
# # awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# # awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${outdir}/${ct}/merge_dmr.1kb.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed

# scenicplus grn_inference motif_enrichment_cistarget \
#     --region_set_folder ${outdir}/${ct}/bed \
#     --cistarget_db_fname ${outdir}/${ct}/merge_dmr.1kb.regions_vs_motifs.rankings.feather \
#     --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.1kb.hdf5 \
#     --temp_dir ${outdir}/${ct}/tmp \
#     --species "homo_sapiens" \
#     --fr_overlap_w_ctx_db 0.4 \
#     --auc_threshold 0.005 \
#     --nes_threshold 3.0 \
#     --rank_threshold 0.05 \
#     --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
#     --annotation_version "v10nr_clust" \
#     --motif_similarity_fdr 0.001 \
#     --orthologous_identity_threshold 0.0 \
#     --annotations_to_use "Direct_annot Orthology_annot" \
#     --write_html \
#     --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.1kb.html \
#     --n_cpu 32

# # for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.1kb.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; echo; done | cut -f6 | sort -k1,1 -u | less
# ) done



In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# for i in `seq 8 34`; do (
# ct=c${i}
# loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
# dmrdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/DMR
# outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_dmr_motif
# mkdir -p ${outdir}/${ct}/bed/loop_dmr/ ${outdir}/${ct}/bed/summit_dmr/
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_dmr/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${dmrdir}/overlap_atac/majortype_36groups/${ct}_merged.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_dmr/${ct}.bed

# scenicplus grn_inference motif_enrichment_cistarget \
#     --region_set_folder ${outdir}/${ct}/bed \
#     --cistarget_db_fname ${outdir}/${ct}/merge_dmr.regions_vs_motifs.rankings.feather \
#     --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
#     --temp_dir ${outdir}/${ct}/tmp \
#     --species "homo_sapiens" \
#     --fr_overlap_w_ctx_db 0.4 \
#     --auc_threshold 0.005 \
#     --nes_threshold 3.0 \
#     --rank_threshold 0.05 \
#     --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
#     --annotation_version "v10nr_clust" \
#     --motif_similarity_fdr 0.001 \
#     --orthologous_identity_threshold 0.0 \
#     --annotations_to_use "Direct_annot Orthology_annot" \
#     --write_html \
#     --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
#     --n_cpu 32

# # for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; done | cut -f6 | sort -k1,1 -u | less
# ) done



In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ## ATAC 1kb createdb

# ct=c0
# REGION_BED=/large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype/${ct}.bed
# GENOME_FASTA=/large_storage/zhoulab/ref/hg38/fasta/hg38.fa
# CHROMSIZES=/large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes
# DATABASE_PREFIX=merge_peak
# SCRIPT_DIR=/home/zhoujt/software/create_cisTarget_databases
# OUT_DIR=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif/${ct}
# mkdir $OUT_DIR
# awk '{printf("%s\t%d\t%d\n",$1,($2+$3)/2,($2+$3)/2)}' $REGION_BED | bedtools slop -i - -b 500 -g /large_storage/zhoulab/ref/hg38/fasta/hg38.main.chrom.sizes > ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.bed
# ${SCRIPT_DIR}/create_fasta_with_padded_bg_from_bed.sh \
#         ${GENOME_FASTA} \
#         ${CHROMSIZES} \
#         ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.bed \
#         ${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa \
#         0 \
#         yes

# export PATH=$PATH:~/software/
# REF_DIR=/large_storage/zhoulab/ref/aertslab_motif_colleciton
# CBDIR=${REF_DIR}/v10nr_clust_public/singletons
# FASTA_FILE=${OUT_DIR}/${DATABASE_PREFIX}.slop1kb.fa
# MOTIF_LIST=${REF_DIR}/motifs.txt

# "${SCRIPT_DIR}/create_cistarget_motif_databases.py" \
#     -f ${FASTA_FILE} \
#     -M ${CBDIR} \
#     -m ${MOTIF_LIST} \
#     -o ${OUT_DIR}/${DATABASE_PREFIX} \
#     --bgpadding 0 \
#     -t 32




In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ## summit/loop ATAC peak enriched motif

# for i in `seq 3 34`; do (
# ct=c${i}
# loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
# atacdir=/large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype
# outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif
# mkdir -p ${outdir}/${ct}/bed/loop_peak/ ${outdir}/${ct}/bed/summit_peak/
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_peak/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/${ct}/${ct}.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_peak/${ct}.bed

# scenicplus grn_inference motif_enrichment_cistarget \
#     --region_set_folder ${outdir}/${ct}/bed \
#     --cistarget_db_fname ${outdir}/${ct}/merge_peak.regions_vs_motifs.rankings.feather \
#     --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
#     --temp_dir ${outdir}/${ct}/tmp \
#     --species "homo_sapiens" \
#     --fr_overlap_w_ctx_db 0.4 \
#     --auc_threshold 0.005 \
#     --nes_threshold 3.0 \
#     --rank_threshold 0.05 \
#     --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
#     --annotation_version "v10nr_clust" \
#     --motif_similarity_fdr 0.001 \
#     --orthologous_identity_threshold 0.0 \
#     --annotations_to_use "Direct_annot Orthology_annot" \
#     --write_html \
#     --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
#     --n_cpu 32
# )
# done

# for f in $(grep "<th>" ${outdir}/${ct}/ctx_results.html | sed 's|<th>||g' | sed 's|</th>||g'); do grep $f /large_storage/zhoulab/ref/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl; done | cut -f6 | sort -k1,1 -u | less



In [ ]:
# [pipeline/shell command — commented out; output is cached on disk]
# ct=c35
# loopdir=/large_storage/zhoulab/zhoujt/project/ENTEx/loop/majortype
# atacdir=/large_storage/zhoulab/zhoujt/project/ENTEx/scATAC/peak/majortype
# outdir=/large_storage/zhoulab/zhoujt/project/ENTEx/analysis/loop_peak_motif
# mkdir -p ${outdir}/${ct}/bed/loop_peak/ ${outdir}/${ct}/bed/summit_peak/
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/loop_peak/${ct}.bed
# awk '{printf("%s\t%d\t%d\n%s\t%d\t%d\n",$1,$2,$3,$4,$5,$6)}' ${loopdir}/c7/c7.loop_summit.bedpe | sort -k1,1 -k2,2n -u | bedtools closest -a ${atacdir}/${ct}.bed -b - -d | awk '$7<1000 {printf("%s\t%d\t%d\n",$1,$2,$3)}' | sort -k1,1 -k2,2n -u > ${outdir}/${ct}/bed/summit_peak/${ct}.bed

# scenicplus grn_inference motif_enrichment_cistarget \
#     --region_set_folder ${outdir}/${ct}/bed \
#     --cistarget_db_fname ${outdir}/${ct}/merge_peak.regions_vs_motifs.rankings.feather \
#     --output_fname_cistarget_result ${outdir}/${ct}/ctx_results.hdf5 \
#     --temp_dir ${outdir}/${ct}/tmp \
#     --species "homo_sapiens" \
#     --fr_overlap_w_ctx_db 0.4 \
#     --auc_threshold 0.005 \
#     --nes_threshold 3.0 \
#     --rank_threshold 0.05 \
#     --path_to_motif_annotations f"{REF_ROOT}/aertslab_motif_colleciton/v10nr_clust_public/snapshots/motifs-v10-nr.hgnc-m0.00001-o0.0.tbl" \
#     --annotation_version "v10nr_clust" \
#     --motif_similarity_fdr 0.001 \
#     --orthologous_identity_threshold 0.0 \
#     --annotations_to_use "Direct_annot Orthology_annot" \
#     --write_html \
#     --output_fname_cistarget_html ${outdir}/${ct}/ctx_results.html \
#     --n_cpu 32
# )
# done

